In [1]:
import os
import numpy as np
import pandas as pd
import re
import subprocess
import tempfile
import random
import time
from collections import defaultdict

# Set Random Seed
random.seed(42)
np.random.seed(42)

dataset_path = "U2AF2"

# Set Default Parameters
path_to_output = dataset_path
kmer = 3
max_num = 3000
test_ratio = 0.1
eval_ratio = 0.1
use_rnafold = True

# Define the FASTA files to read directly
positive_fasta = f"{dataset_path}/{dataset_path}.positive.fa"
negative_fasta = f"{dataset_path}/{dataset_path}.negative.fa"

OUTPUT_FILE = 'original.tsv'

print("Parameter settings completed")

Parameter settings completed


In [2]:
def run_rnafold_robust(sequence, max_retries=5):
    rna_sequence = sequence
    
    for attempt in range(max_retries):
        # Create a temporary directory using tempfile and ensure automatic cleanup.
        with tempfile.TemporaryDirectory() as temp_dir:
            try:
                # Create a temporary FASTA file
                fasta_file = os.path.join(temp_dir, "input.fa")
                with open(fasta_file, 'w') as f:
                    f.write(f">seq\n{rna_sequence}\n")
                
                # Run RNAfold
                cmd = ['RNAfold', '-p', '--noPS']
                result = subprocess.run(
                    cmd,
                    stdin=open(fasta_file, 'r'),
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    text=True,
                    timeout=60,
                    cwd=temp_dir  # Run the command in the temporary directory
                )
                
                # Check whether RNAfold has been successfully executed
                if result.returncode != 0:
                    error_msg = f"RNAfold failed with return code {result.returncode}: {result.stderr}"
                    if attempt < max_retries - 1:
                        time.sleep(1)  # Retry after a short delay
                        continue
                    else:
                        raise Exception(error_msg)
                
                # Check if the bitmap file has been generated
                dp_file = os.path.join(temp_dir, "seq_dp.ps")
                if not os.path.exists(dp_file):
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
                    else:
                        raise FileNotFoundError(f"RNAfold dot plot file not found after {max_retries} attempts")
                
                # Parse secondary structure from standard output
                structure = parse_structure_from_output(result.stdout, len(rna_sequence))
                
                if len(structure) != len(rna_sequence):
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
                    else:
                        raise ValueError(f"Structure length mismatch: expected {len(rna_sequence)}, got {len(structure)}")
                
                # Read the pairing probability file
                probabilities = parse_dot_plot_file_correct(dp_file, len(rna_sequence))
                
                if len(probabilities) != len(rna_sequence):
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
                if attempt > 0:
                    print(f"Successfully processed sequence after {attempt + 1} attempts")
                return structure, probabilities
                
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(1)
                else:
                    print(f"All {max_retries} attempts failed for sequence: {e}")
                    raise
    
    # If all attempts fail, throw an exception
    raise Exception(f"Failed to process sequence after {max_retries} attempts")

def parse_dot_plot_file_correct(dp_file, seq_length):
    with open(dp_file, 'r') as f:
        content = f.read()

    # Initialize the probability array
    probs = [0.0] * seq_length

    # parse UBOX data
    lines = content.split('\n')
    ubox_count = 0

    # Search for the UBOX data section
    for i, line in enumerate(lines):
        line = line.strip()

        # Skip blank lines and comments
        if not line or line.startswith('%'):
            continue

        # Search for UBOX data rows
        parts = line.split()
        if len(parts) >= 4 and parts[3] == 'ubox':
            i_pos = int(parts[0]) - 1  # Convert to 0-based index
            j_pos = int(parts[1]) - 1
            prob = float(parts[2])

            if 0 <= i_pos < seq_length and 0 <= j_pos < seq_length:
                # Cumulative Probability
                probs[i_pos] = max(probs[i_pos], prob)
                probs[j_pos] = max(probs[j_pos], prob)
                ubox_count += 1

    return probs

def parse_structure_from_output(output, seq_length):
    lines = output.split('\n')

    # Search for secondary structure lines
    structure = '.' * seq_length
    for line in lines:
        line = line.strip()
        if len(line) > 0 and not line.startswith('>') and not line.startswith(' '):
            parts = line.split()
            for part in parts:
                if (len(part) == seq_length and
                        all(c in '().' for c in part)):
                    structure = part
                    break

    return structure

print("The RNAfold function has been defined successfully.")

The RNAfold function has been defined successfully.


In [3]:
def encode_structure(structure_string):
    """Encode the secondary structure string into a numerical form"""
    encoding_map = {'.': 0, '(': 1, ')': 2}
    encoded = []
    for char in structure_string:
        if char in encoding_map:
            encoded.append(encoding_map[char])
        else:
            encoded.append(0)  # Default Value
    return encoded

def seq2kmer(seq, k):
    """Convert the original sequence to kmer"""
    kmer = [seq[x:x + k] for x in range(len(seq) + 1 - k)]
    kmers = " ".join(kmer)
    return kmers

def split_dataset_balanced(sequences, test_ratio=0.2, eval_ratio=0.2):
    """Balance the dataset and ensure that each subset contains both positive and negative samples."""
    # First, shuffle the entire sequence list
    random.shuffle(sequences)
    
    # Separate positive and negative samples
    positive_samples = [seq for seq in sequences if seq[1] == 1]
    negative_samples = [seq for seq in sequences if seq[1] == 0]
    
    # Calculate the size of each set
    total_positive = len(positive_samples)
    total_negative = len(negative_samples)
    
    # Calculate the number of positive and negative samples for each set
    test_positive_count = max(1, int(total_positive * test_ratio))
    test_negative_count = max(1, int(total_negative * test_ratio))
    
    eval_positive_count = max(1, int(total_positive * eval_ratio))
    eval_negative_count = max(1, int(total_negative * eval_ratio))
    
    # Segmentation of positive samples
    test_positive = positive_samples[:test_positive_count]
    eval_positive = positive_samples[test_positive_count:test_positive_count+eval_positive_count]
    train_positive = positive_samples[test_positive_count+eval_positive_count:]
    
    # Divide the negative samples
    test_negative = negative_samples[:test_negative_count]
    eval_negative = negative_samples[test_negative_count:test_negative_count+eval_negative_count]
    train_negative = negative_samples[test_negative_count+eval_negative_count:]
    
    # Merge and shuffle
    test_sequences = test_positive + test_negative
    eval_sequences = eval_positive + eval_negative
    train_sequences = train_positive + train_negative
    
    random.shuffle(test_sequences)
    random.shuffle(eval_sequences)
    random.shuffle(train_sequences)
    
    return train_sequences, eval_sequences, test_sequences

print("Tool function definition completed")

Tool function definition completed


In [4]:
def read_fasta_file(file_path, label):
    sequences = []
    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} does not exist")
        return sequences
        
    with open(file_path, 'r') as f:
        current_sequence = ""
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_sequence and len(current_sequence) >= kmer:
                    sequences.append((current_sequence, label))
                current_sequence = ""
            else:
                current_sequence += line
        # Add the last sequence
        if current_sequence and len(current_sequence) >= kmer:
            sequences.append((current_sequence, label))
    return sequences

def create_dataset_files():
    print(f"Start processing FASTA files directly")
    
    # Read positive and negative sequences directly from FASTA files
    positive_sequences = read_fasta_file(positive_fasta, 1)
    negative_sequences = read_fasta_file(negative_fasta, 0)
    
    print(f"Positive sequences: {len(positive_sequences)}")
    print(f"Negative sequences: {len(negative_sequences)}")
    
    if len(positive_sequences) == 0 or len(negative_sequences) == 0:
        print("Error: No sequences found in one or both FASTA files")
        return

    # Randomly select max_num/2 sequences from both the positive and negative samples
    sample_per_class = max_num // 2
    
    # If the number of a certain type of sample is insufficient, then use all the samples
    sampled_positive = random.sample(positive_sequences, min(sample_per_class, len(positive_sequences)))
    sampled_negative = random.sample(negative_sequences, min(sample_per_class, len(negative_sequences)))
    
    # Merge positive and negative samples
    balanced_sequences = sampled_positive + sampled_negative
    random.shuffle(balanced_sequences)
    
    print(f"After sampling: Positive samples {len(sampled_positive)}, Negative samples {len(sampled_negative)}, Total {len(balanced_sequences)}")
    
    # Divide the dataset
    total_count = len(balanced_sequences)
    test_count = int(total_count * test_ratio)
    eval_count = int(total_count * eval_ratio)
    
    test_sequences = balanced_sequences[:test_count]
    eval_sequences = balanced_sequences[test_count:test_count+eval_count]
    train_sequences = balanced_sequences[test_count+eval_count:]
    
    # Shuffle the data set again for each one
    random.shuffle(train_sequences)
    random.shuffle(eval_sequences)
    random.shuffle(test_sequences)
    
    print(f"Split result: Training set {len(train_sequences)}, Validation set {len(eval_sequences)}, Test set {len(test_sequences)}")
    
    # Create files for each dataset
    datasets = [
    (train_sequences, 'train.tsv'),
    (eval_sequences, 'eval.tsv'),
    (test_sequences, 'test.tsv')
    ]
    
    processed_count = 0
    failed_sequences = 0
    
    for seq_list, file_name in datasets:
        if seq_list:
            output_dir = os.path.join(path_to_output)
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, file_name)

            # Create a new file
            with open(output_path, mode='w') as f:
                if use_rnafold:
                    f.write('sequence\tlabel\tstructure\tpairing_probabilities\n')
                else:
                    f.write('sequence\tlabel\n')

                # Progress display
                total_seqs = len(seq_list)
                print(f"Processing {file_name}: {total_seqs} sequences")
                
                for idx, (sequence, label) in enumerate(seq_list):
                    # The progress is displayed once for every 100 sequences processed
                    if idx % 100 == 0:
                        print(f"  Progress: {idx}/{total_seqs} sequences processed")
                        
                    kmer_sequence = seq2kmer(sequence, kmer)

                    if use_rnafold:
                        try:
                            structure, pairing_probs = run_rnafold_robust(sequence)
                            structure_encoding = encode_structure(structure)

                            # Make sure the length is consistent
                            if len(structure_encoding) != len(sequence):
                                structure_encoding = structure_encoding[:len(sequence)] if len(
                                    structure_encoding) > len(sequence) else structure_encoding + [0] * (
                                        len(sequence) - len(structure_encoding))
                            if len(pairing_probs) != len(sequence):
                                pairing_probs = pairing_probs[:len(sequence)] if len(pairing_probs) > len(
                                    sequence) else pairing_probs + [0.0] * (len(sequence) - len(pairing_probs))

                            structure_str = ','.join(map(str, structure_encoding))
                            pairing_str = ','.join(f"{max(0, prob):.3f}" for prob in pairing_probs)
                            f.write(f"{kmer_sequence}\t{label}\t{structure_str}\t{pairing_str}\n")
                        except Exception as e:
                            print(f"CRITICAL ERROR: Failed to process sequence after all retries: {str(e)}")
                            failed_sequences += 1
                            print(f"Skipping sequence due to persistent RNAfold failure")
                    else:
                        f.write(f"{kmer_sequence}\t{label}\n")
            
            # Verify the output file
            with open(output_path, 'r') as f:
                lines = f.readlines()
                if len(lines) > 1:  # Skip the title row
                    labels = [line.split('\t')[1] for line in lines[1:]]
                    pos_count = sum(1 for label in labels if label == '1')
                    neg_count = sum(1 for label in labels if label == '0')
                    print(f"  Output file {file_name}: {len(labels)} sequences, positive samples {pos_count}, negative samples {neg_count}")

            processed_count += len(seq_list)

    print(f"Processing completed! A total of {processed_count} sequences were processed.")
    if failed_sequences > 0:
        print(f"WARNING: {failed_sequences} sequences failed to process and were skipped.")
    return

print("The main processing function definition is complete.")

The main processing function definition is complete.


In [5]:
# Execute the main function
print("Start processing the dataset...")
create_dataset_files()
print("All processing is complete！")

Start processing the dataset...
Start processing FASTA files directly
Positive sequences: 60000
Negative sequences: 60000
After sampling: Positive samples 1500, Negative samples 1500, Total 3000
Split result: Training set 2400, Validation set 300, Test set 300
Processing train.tsv: 2400 sequences
  Progress: 0/2400 sequences processed
  Progress: 100/2400 sequences processed
  Progress: 200/2400 sequences processed
  Progress: 300/2400 sequences processed
  Progress: 400/2400 sequences processed
  Progress: 500/2400 sequences processed
  Progress: 600/2400 sequences processed
  Progress: 700/2400 sequences processed
  Progress: 800/2400 sequences processed
  Progress: 900/2400 sequences processed
  Progress: 1000/2400 sequences processed
  Progress: 1100/2400 sequences processed
  Progress: 1200/2400 sequences processed
  Progress: 1300/2400 sequences processed
  Progress: 1400/2400 sequences processed
  Progress: 1500/2400 sequences processed
  Progress: 1600/2400 sequences processed
